# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정
9. 보고서용 실험 실행
10. 시각화 및 과적합 확인
11. 하이퍼파라미터 비교 실험
12. 전체 테스트와 제출 전 확인

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.


## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 보고서용 실험 실행

아래 셀들은 `REPORT.md`에 적을 BPE, 사전 학습, 미세 조정 결과를 수집합니다. Colab GPU에서 실행하는 것을 기준으로 하며, 테스트는 실행하지 않습니다.


In [ ]:
# 보고서용 실험 설정: Basic 기본값
import json
import random
import time
import platform
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from bpe import BPETokenizer
from dataset import create_dataloader
from model import GPTModel
from train import calc_loss_batch, evaluate_model, generate
from finetune import (
    ReviewSentimentDataset,
    GPTForSequenceClassification,
    train_epoch_sentiment,
    evaluate_sentiment,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPORT_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", REPORT_DEVICE)
if REPORT_DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

REPORT_PATHS = {
    "vocab": repo_dir / "data" / "vocab_bpe_basic_3000.json",
    "checkpoint": repo_dir / "checkpoints" / "report_basic_last.pt",
    "sentiment_train": repo_dir / "data" / "nsmc_sentiment_train.jsonl",
    "sentiment_val": repo_dir / "data" / "nsmc_sentiment_val.jsonl",
    "sentiment_test": repo_dir / "data" / "nsmc_sentiment_test.jsonl",
}

REPORT_BPE = {
    "vocab_size": 3000,
    "corpus_limit": 1_500_000,
}

REPORT_MODEL_CONFIG = {
    "vocab_size": REPORT_BPE["vocab_size"],
    "context_length": 128,
    "emb_dim": 192,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

REPORT_PRETRAIN = {
    "batch_size": 8,
    "num_epochs": 1,
    "lr": 3e-4,
    "weight_decay": 0.01,
    "eval_freq": 100,
    "eval_iter": 20,
    "start_context": "이 영화는",
    "max_new_tokens": 50,
    "temperature": 0.8,
    "top_k": 40,
}

REPORT_FINETUNE = {
    "max_length": 128,
    "batch_size": 16,
    "num_epochs": 1,
    "backbone_lr": 1e-4,
    "classifier_lr": 3e-4,
    "drop_rate": 0.1,
}


def read_jsonl(path: str | Path) -> list[dict]:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def count_parameters(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def fmt_seconds(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}초"
    return f"{seconds / 60:.1f}분"


def compact_text(text: str, max_len: int = 180) -> str:
    text = " ".join(str(text).split())
    return text if len(text) <= max_len else text[: max_len - 3] + "..."


def decode_token_ids_safe(tokenizer, token_ids: list[int]) -> str:
    """생성된 token ID를 UTF-8 오류로 중단되지 않게 문자열로 바꿉니다."""
    try:
        return tokenizer.decode(token_ids)
    except UnicodeDecodeError:
        special_ids = {
            tokenizer.get_pad_id(),
            tokenizer.get_unk_id(),
            tokenizer.get_bos_id(),
            tokenizer.get_eos_id(),
        }
        byte_values = []
        for token_id in token_ids:
            if token_id in special_ids:
                continue
            try:
                byte_values.extend(tokenizer.token_to_bytes(token_id))
            except (KeyError, ValueError):
                continue
        return bytes(byte_values).decode("utf-8", errors="replace")

for name, value in REPORT_PATHS.items():
    print(f"{name}: {value}")


In [ ]:
# BPE vocabulary 학습 또는 로드
if not corpus:
    raise RuntimeError("LM train corpus가 비어 있습니다. 먼저 NSMC 데이터 준비 셀을 실행하세요.")

bpe_corpus = corpus[: REPORT_BPE["corpus_limit"]]
report_tokenizer = BPETokenizer(vocab_size=REPORT_BPE["vocab_size"])

bpe_started = time.perf_counter()
if REPORT_PATHS["vocab"].exists():
    report_tokenizer.load(REPORT_PATHS["vocab"])
    report_bpe_mode = "loaded"
else:
    REPORT_PATHS["vocab"].parent.mkdir(parents=True, exist_ok=True)
    report_tokenizer.train(bpe_corpus)
    report_tokenizer.save(REPORT_PATHS["vocab"])
    report_bpe_mode = "trained"
report_bpe_elapsed = time.perf_counter() - bpe_started

restore_samples = [
    "이 영화는 정말 좋았다!",
    "연기, 음악, 연출 모두 좋음 ㅋㅋ",
    "한글 English 123!?",
]
report_bpe_restore = []
for text in restore_samples:
    decoded = report_tokenizer.decode(report_tokenizer.encode(text, add_bos_eos=True))
    report_bpe_restore.append({"text": text, "decoded": decoded, "ok": decoded == text})

print("BPE mode:", report_bpe_mode)
print("corpus chars:", len(bpe_corpus))
print("target vocab_size:", REPORT_BPE["vocab_size"])
print("actual vocab size:", len(report_tokenizer.id_to_token))
print("elapsed:", fmt_seconds(report_bpe_elapsed))
print("vocab path:", REPORT_PATHS["vocab"])
for row in report_bpe_restore:
    print(f"restore ok={row['ok']}: {row['decoded']}")


## 10. 생성형 모델 Colab 실행

아래 셀은 `data/vocab_bpe_basic_3000.json`을 로드해서 첫 생성형 GPT 모델을 학습하고, 영화 리뷰 형태의 생성 결과와 train/test loss를 수집합니다. 여기서 test loss는 NSMC의 LM validation 텍스트(`nsmc_lm_val.txt`)에서 계산한 hold-out loss입니다.


In [ ]:
# Colab 보고서용 생성형 모델 사전학습 준비
# - 이미 만들어 둔 data/vocab_bpe_basic_3000.json을 우선 로드합니다.
# - Colab 시간이 부족하면 *_CHAR_LIMIT, num_epochs, eval_freq를 줄이세요.

if not corpus or not val_corpus:
    raise RuntimeError("LM train/test corpus가 필요합니다. 먼저 NSMC 데이터 준비 셀을 실행하세요.")
if not REPORT_PATHS["vocab"].exists():
    raise FileNotFoundError(f"어휘사전 JSON이 없습니다: {REPORT_PATHS['vocab']}")

report_tokenizer = BPETokenizer(vocab_size=REPORT_BPE["vocab_size"])
report_tokenizer.load(REPORT_PATHS["vocab"])
REPORT_MODEL_CONFIG["vocab_size"] = len(report_tokenizer.id_to_token)
print("loaded vocab:", REPORT_PATHS["vocab"])
print("actual vocab size:", len(report_tokenizer.id_to_token))

REPORT_LM_TRAIN_CHAR_LIMIT = 1_500_000
REPORT_LM_TEST_CHAR_LIMIT = 160_000

lm_train_text = corpus[:REPORT_LM_TRAIN_CHAR_LIMIT]
lm_test_text = val_corpus[:REPORT_LM_TEST_CHAR_LIMIT]

tokenize_started = time.perf_counter()
report_train_token_ids = report_tokenizer.encode(lm_train_text)
report_test_token_ids = report_tokenizer.encode(lm_test_text)
report_tokenize_elapsed = time.perf_counter() - tokenize_started

report_train_loader = create_dataloader(
    report_train_token_ids,
    context_length=REPORT_MODEL_CONFIG["context_length"],
    batch_size=REPORT_PRETRAIN["batch_size"],
    stride=REPORT_MODEL_CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0,
)
report_test_loader = create_dataloader(
    report_test_token_ids,
    context_length=REPORT_MODEL_CONFIG["context_length"],
    batch_size=REPORT_PRETRAIN["batch_size"],
    stride=REPORT_MODEL_CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0,
)

if len(report_train_loader) == 0 or len(report_test_loader) == 0:
    raise RuntimeError("DataLoader가 비어 있습니다. corpus limit와 context_length를 확인하세요.")

report_gpt = GPTModel(REPORT_MODEL_CONFIG).to(REPORT_DEVICE)
report_optimizer = torch.optim.AdamW(
    report_gpt.parameters(),
    lr=REPORT_PRETRAIN["lr"],
    weight_decay=REPORT_PRETRAIN["weight_decay"],
)
report_model_param_count = count_parameters(report_gpt)

print("train chars:", f"{len(lm_train_text):,}")
print("test chars:", f"{len(lm_test_text):,}")
print("train tokens:", f"{len(report_train_token_ids):,}")
print("test tokens:", f"{len(report_test_token_ids):,}")
print("train batches:", f"{len(report_train_loader):,}")
print("test batches:", f"{len(report_test_loader):,}")
print("tokenization elapsed:", fmt_seconds(report_tokenize_elapsed))
print("model parameters:", f"{report_model_param_count:,}")


In [ ]:
# 생성형 GPT 사전학습 실행 및 영화 리뷰 생성 결과 확인
report_pretrain_history = []
report_pretrain_samples = []
report_tokens_seen = 0
report_global_step = 0
pretrain_started = time.perf_counter()

for epoch in range(1, REPORT_PRETRAIN["num_epochs"] + 1):
    report_gpt.train()
    for input_batch, target_batch in report_train_loader:
        report_optimizer.zero_grad()
        loss = calc_loss_batch(input_batch, target_batch, report_gpt, REPORT_DEVICE)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(report_gpt.parameters(), max_norm=1.0)
        report_optimizer.step()

        report_tokens_seen += input_batch.numel()
        report_global_step += 1

        if report_global_step % REPORT_PRETRAIN["eval_freq"] == 0:
            train_loss, test_loss = evaluate_model(
                report_gpt,
                report_train_loader,
                report_test_loader,
                REPORT_DEVICE,
                REPORT_PRETRAIN["eval_iter"],
            )
            row = {
                "epoch": epoch,
                "step": report_global_step,
                "tokens_seen": report_tokens_seen,
                "train_loss": train_loss,
                "test_loss": test_loss,
                "elapsed_sec": time.perf_counter() - pretrain_started,
            }
            report_pretrain_history.append(row)
            print(
                f"epoch={epoch} step={report_global_step} "
                f"train_loss={train_loss:.4f} test_loss={test_loss:.4f} "
                f"elapsed={fmt_seconds(row['elapsed_sec'])}"
            )

    train_loss, test_loss = evaluate_model(
        report_gpt,
        report_train_loader,
        report_test_loader,
        REPORT_DEVICE,
        REPORT_PRETRAIN["eval_iter"],
    )
    row = {
        "epoch": epoch,
        "step": report_global_step,
        "tokens_seen": report_tokens_seen,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "elapsed_sec": time.perf_counter() - pretrain_started,
    }
    report_pretrain_history.append(row)
    print(
        f"epoch_end={epoch} step={report_global_step} "
        f"train_loss={train_loss:.4f} test_loss={test_loss:.4f} "
        f"elapsed={fmt_seconds(row['elapsed_sec'])}"
    )

report_pretrain_elapsed = time.perf_counter() - pretrain_started
REPORT_PATHS["checkpoint"].parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state_dict": report_gpt.state_dict(),
        "model_config": REPORT_MODEL_CONFIG,
        "history": report_pretrain_history,
        "tokens_seen": report_tokens_seen,
        "global_step": report_global_step,
    },
    REPORT_PATHS["checkpoint"],
)
print("checkpoint saved:", REPORT_PATHS["checkpoint"])
print("pretrain elapsed:", fmt_seconds(report_pretrain_elapsed))

review_prompts = [
    "이 영화는",
    "배우들의 연기는",
    "스토리는",
]

report_gpt.eval()
for prompt in review_prompts:
    start_ids = torch.tensor(
        report_tokenizer.encode(prompt),
        dtype=torch.long,
        device=REPORT_DEVICE,
    ).unsqueeze(0)
    with torch.no_grad():
        sampled_ids = generate(
            model=report_gpt,
            idx=start_ids,
            max_new_tokens=REPORT_PRETRAIN["max_new_tokens"],
            context_size=REPORT_MODEL_CONFIG["context_length"],
            temperature=REPORT_PRETRAIN["temperature"],
            top_k=REPORT_PRETRAIN["top_k"],
            eos_id=report_tokenizer.get_eos_id(),
        )
    sample_text = decode_token_ids_safe(report_tokenizer, sampled_ids.squeeze(0).tolist())
    report_pretrain_samples.append({"prompt": prompt, "text": sample_text})
    print("=" * 80)
    print(f"prompt: {prompt}")
    print(compact_text(sample_text, max_len=500))


## 11. Train/Test Loss 시각화

학습 중 수집한 train loss와 test loss를 같은 축에 그려 과대적합 여부를 확인합니다. test loss가 train loss보다 크게 벌어지면 일반화 격차가 커진 것으로 해석할 수 있습니다.


In [ ]:
# matplotlib 기반 train loss / test loss 과대적합 확인 시각화
import matplotlib.pyplot as plt

if "report_pretrain_history" not in globals() or not report_pretrain_history:
    raise RuntimeError("먼저 생성형 GPT 사전학습 실행 셀을 완료하세요.")

steps = [row["step"] for row in report_pretrain_history]
train_losses = [row["train_loss"] for row in report_pretrain_history]
test_losses = [row["test_loss"] for row in report_pretrain_history]
gaps = [test - train for train, test in zip(train_losses, test_losses)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(steps, train_losses, marker="o", label="Train loss")
axes[0].plot(steps, test_losses, marker="o", label="Test loss")
axes[0].set_title("Generative GPT Loss Curve")
axes[0].set_xlabel("Training step")
axes[0].set_ylabel("Cross entropy loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(steps, gaps, marker="o", color="#E45756", label="Test - Train")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Overfitting Gap")
axes[1].set_xlabel("Training step")
axes[1].set_ylabel("Loss gap")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

final_row = report_pretrain_history[-1]
print("REPORT.md 기록용")
print("- model params:", f"{report_model_param_count:,}")
print("- final train loss:", f"{final_row['train_loss']:.4f}")
print("- final test loss:", f"{final_row['test_loss']:.4f}")
print("- final gap(test-train):", f"{final_row['test_loss'] - final_row['train_loss']:.4f}")
print("- tokens seen:", f"{final_row['tokens_seen']:,}")
print("- elapsed:", fmt_seconds(report_pretrain_elapsed))
print("- checkpoint:", REPORT_PATHS["checkpoint"])

print("\n생성 샘플")
for sample in report_pretrain_samples:
    print(f"[{sample['prompt']}] {compact_text(sample['text'], max_len=300)}")


## 12. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.


In [ ]:
run_pytest("tests/")